# 03 - Data Cleaning and Preparation (Objective 3)
Output: a clean NLP dataset with **two text versions** and train/validation/test splits.

| Column | Used for |
|---|---|
| `text_light` | BERT, BERTopic (keeps punctuation and negations) |
| `text_heavy` | TF-IDF baselines only (lowercased, stop-words removed, lemmatised) |

In [ ]:
%pip install -q pandas scikit-learn nltk emoji langdetect pyarrow

In [ ]:
import re, html
import pandas as pd
import numpy as np
import emoji
import nltk
from pathlib import Path
from langdetect import detect, DetectorFactory
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from sklearn.model_selection import train_test_split

nltk.download("wordnet", quiet=True); nltk.download("omw-1.4", quiet=True)
DetectorFactory.seed = 0
Path("data/processed").mkdir(parents=True, exist_ok=True)

df = pd.read_parquet("data/raw/reviews_raw.parquet")
print("Start:", len(df))

## 1. Structured data cleaning

In [ ]:
df = df.drop_duplicates()                                   # exact duplicate rows
df = df.drop_duplicates(subset="review_id")                 # duplicate IDs
df = df.dropna(subset=["review_id", "review_text", "rating"])  # missing identifiers / text
df = df[df["rating"].between(1, 5)]                         # invalid ratings

for col in ["review_date", "check_in", "check_out"]:
    df[col] = pd.to_datetime(df[col], errors="coerce")
df = df[df["review_date"].notna() & (df["review_date"] <= pd.Timestamp.today())]   # bad / future dates
df = df[df["check_out"] >= df["check_in"]]                  # inconsistent dates

print("After structured cleaning:", len(df))

## 2. Text cleaning (two versions)

In [ ]:
lemmatizer = WordNetLemmatizer()
KEEP = {"no", "not", "nor", "never", "n't", "but", "very", "too"}     # do NOT remove negations!
STOPWORDS = set(ENGLISH_STOP_WORDS) - KEEP

def light_clean(t):
    t = html.unescape(str(t))
    t = re.sub(r"<[^>]+>", " ", t)                       # HTML tags
    t = re.sub(r"http\S+|www\.\S+", " ", t)             # URLs
    t = emoji.demojize(t, delimiters=(" ", " "))         # emoji -> words
    t = re.sub(r"\s+", " ", t).strip()                   # whitespace
    return t

def heavy_clean(t):
    t = light_clean(t).lower()
    t = re.sub(r"[^a-z\s]", " ", t)                      # drop punctuation/digits
    words = [lemmatizer.lemmatize(w) for w in t.split() if w not in STOPWORDS and len(w) > 2]
    return " ".join(words)

df["text_light"] = df["review_text"].apply(light_clean)
df["text_heavy"] = df["review_text"].apply(heavy_clean)

df[["review_text", "text_light", "text_heavy"]].head(3)

In [ ]:
# remove empty/very short reviews and keep English only
df["n_words"] = df["text_light"].str.split().str.len()
df = df[df["n_words"] >= 3]

def is_english(t):
    try: return detect(t) == "en"
    except Exception: return False

df["is_english"] = df["text_light"].apply(is_english)
print("English share:", df["is_english"].mean().round(3))
df = df[df["is_english"]].drop(columns="is_english")
print("After text cleaning:", len(df))

## 3. Sentiment labels
Mapping rule (version this rule with your code!): **1-2 = negative, 3 = neutral, 4-5 = positive**.

> Spot-check 200-300 random reviews by hand to confirm the mapping is trustworthy.

In [ ]:
def rating_to_label(r):
    return "negative" if r <= 2 else "neutral" if r == 3 else "positive"

df["label"] = df["rating"].apply(rating_to_label)
print(df["label"].value_counts(normalize=True).round(3))

# manual spot-check sample -> open the CSV, read, and mark correct/incorrect
df.sample(min(250, len(df)), random_state=42)[["review_id", "rating", "label", "review_text"]] \
  .to_csv("data/processed/label_spot_check.csv", index=False)

## 4. Data quality validation
Simple checks that always run (and stop the pipeline if they fail). The Great Expectations version below is optional.

In [ ]:
today = pd.Timestamp.today()
checks = {
    "review_id not null":        df["review_id"].notna().all(),
    "review_id unique":          df["review_id"].is_unique,
    "rating between 1 and 5":    df["rating"].between(1, 5).all(),
    "text_light not empty":      (df["text_light"].str.len() > 0).all(),
    "label in valid set":        df["label"].isin(["negative", "neutral", "positive"]).all(),
    "review_date not in future": (df["review_date"] <= today).all(),
}
for name, ok in checks.items():
    print("PASS" if ok else "FAIL", "-", name)

assert all(checks.values()), "Data quality checks failed - fix before modelling"

In [ ]:
# OPTIONAL: the same checks with Great Expectations (legacy 0.18 API).
# Install first in a separate environment:  pip install "great_expectations<1.0"
try:
    import great_expectations as ge
    gdf = ge.from_pandas(df)
    ge_checks = [
        gdf.expect_column_values_to_not_be_null("review_id"),
        gdf.expect_column_values_to_be_unique("review_id"),
        gdf.expect_column_values_to_be_between("rating", 1, 5),
        gdf.expect_column_values_to_be_in_set("label", ["negative", "neutral", "positive"]),
    ]
    for c in ge_checks:
        print("GE", "PASS" if c["success"] else "FAIL", "-", c["expectation_config"]["expectation_type"])
except Exception as e:
    print("Great Expectations skipped:", type(e).__name__)

## 5. Train / validation / test split (70 / 15 / 15, stratified)
Guests can write several reviews, so we split **by guest** to avoid leakage, then check class balance.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

gss1 = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=42)
train_idx, temp_idx = next(gss1.split(df, groups=df["guest_id"]))
train, temp = df.iloc[train_idx], df.iloc[temp_idx]

gss2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=42)
val_idx, test_idx = next(gss2.split(temp, groups=temp["guest_id"]))
val, test = temp.iloc[val_idx], temp.iloc[test_idx]

for name, part in [("train", train), ("val", val), ("test", test)]:
    print(name, len(part), part["label"].value_counts(normalize=True).round(2).to_dict())
    part.to_parquet(f"data/processed/{name}.parquet", index=False)

df.to_parquet("data/processed/reviews_clean.parquet", index=False)
print("\nSaved clean dataset:", df.shape)